### 1. Load the 12.4G data, filter df_2025, and high related columns

import duckdb

INFILE = r"311_Service_Requests_from_2020_to_Present_20260116.csv"
OUTPARQUET = r"311_2025.parquet"

con = duckdb.connect()

con.execute(f"""
COPY (
  SELECT *
  FROM read_csv_auto(
    '{INFILE}',
    types={{'Incident Zip': 'VARCHAR'}}
  )
  WHERE "Created Date" >= TIMESTAMP '2025-01-01'
    AND "Created Date" <  TIMESTAMP '2026-01-01'
) TO '{OUTPARQUET}' (FORMAT PARQUET);
""")

print("Saved ->", OUTPARQUET)


In [20]:
import pandas as pd
df_2025 = pd.read_parquet("311_2025.parquet")

df_2025["Incident Zip"].head(10), df_2025["Incident Zip"].dtype

(0    10029
 1    10031
 2    11234
 3    11235
 4    10031
 5    11234
 6    11214
 7    11368
 8    11204
 9    11226
 Name: Incident Zip, dtype: object,
 dtype('O'))

In [22]:
df_2025.shape

(3550188, 44)

In [26]:
df_2025.columns

Index(['Unique Key', 'Created Date', 'Closed Date', 'Agency', 'Agency Name',
       'Problem (formerly Complaint Type)',
       'Problem Detail (formerly Descriptor)', 'Additional Details',
       'Location Type', 'Incident Zip', 'Incident Address', 'Street Name',
       'Cross Street 1', 'Cross Street 2', 'Intersection Street 1',
       'Intersection Street 2', 'Address Type', 'City', 'Landmark',
       'Facility Type', 'Status', 'Due Date', 'Resolution Description',
       'Resolution Action Updated Date', 'Community Board', 'Council District',
       'Police Precinct', 'BBL', 'Borough', 'X Coordinate (State Plane)',
       'Y Coordinate (State Plane)', 'Open Data Channel Type',
       'Park Facility Name', 'Park Borough', 'Vehicle Type',
       'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name',
       'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment',
       'Latitude', 'Longitude', 'Location'],
      dtype='object')

In [28]:
df_2025.to_csv("311_2025.csv", index=False)

In [30]:
df_2025.columns

Index(['Unique Key', 'Created Date', 'Closed Date', 'Agency', 'Agency Name',
       'Problem (formerly Complaint Type)',
       'Problem Detail (formerly Descriptor)', 'Additional Details',
       'Location Type', 'Incident Zip', 'Incident Address', 'Street Name',
       'Cross Street 1', 'Cross Street 2', 'Intersection Street 1',
       'Intersection Street 2', 'Address Type', 'City', 'Landmark',
       'Facility Type', 'Status', 'Due Date', 'Resolution Description',
       'Resolution Action Updated Date', 'Community Board', 'Council District',
       'Police Precinct', 'BBL', 'Borough', 'X Coordinate (State Plane)',
       'Y Coordinate (State Plane)', 'Open Data Channel Type',
       'Park Facility Name', 'Park Borough', 'Vehicle Type',
       'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name',
       'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment',
       'Latitude', 'Longitude', 'Location'],
      dtype='object')

##### Load "311_2025.csv"

In [5]:
import pandas as pd

# 1) 直接读取（2GB 级别，建议 low_memory=False）
df_2025 = pd.read_csv("311_2025.csv", low_memory=False)

print("Loaded:", df_2025.shape)
df_2025.head(3)


Loaded: (3550188, 44)


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
0,67351762,2025-12-31 23:59:28,2026-01-01 00:40:32,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10029,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.792141,-73.950097,POINT (-73.950097072847 40.792140506424)
1,67344624,2025-12-31 23:59:23,2026-01-01 01:03:42,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,10031,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.825137,-73.949447,POINT (-73.949447237231 40.82513729001)
2,67346873,2025-12-31 23:59:21,2026-01-01 00:58:13,NYPD,New York City Police Department,Blocked Driveway,Partial Access,NaN,Street/Sidewalk,11234,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.619995,-73.921167,POINT (-73.921167390534 40.619994693785)


In [12]:
use_cols = [
    "Unique Key",
    "Created Date",
    "Closed Date",
    "Status",
    "Agency",
    "Agency Name",
    "Problem (formerly Complaint Type)",
    "Problem Detail (formerly Descriptor)",
    "Location Type",
    "Incident Zip",
    "Incident Address",
    "Borough",
    "Community Board",
    "Council District",
    "Resolution Description",
    "Latitude",
    "Longitude",
]

In [14]:
use_cols = [c for c in use_cols if c in df_2025.columns]
df_map = df_2025[use_cols].dropna(subset=["Latitude","Longitude"]).copy()
df_map.shape

(3499993, 17)

In [16]:
df_2025.to_csv("311_2025.csv", index=False)

### 2. Filter Complaint_Type about Sanitation

In [38]:
import numpy as np

col = "Problem (formerly Complaint Type)"

vals = (
    df_2025[col]
    .dropna()
    .astype(str)
    .unique()
)

vals = sorted(vals)

print(f"Unique {col} count =", len(vals))
print("\n".join(vals))

Unique Problem (formerly Complaint Type) count = 191
AHV Inspection Unit
APPLIANCE
ASBESTOS
Abandoned Bike
Abandoned Vehicle
Air Quality
Animal Facility - No Permit
Animal in a Park
Animal-Abuse
Asbestos
BEST/Site Safety
Beach/Pool/Sauna Complaint
Bench
Bike Rack
Bike/Roller/Skate
Bike/Roller/Skate Chronic
Blocked Driveway
Boilers
Borough Office
Bridge Condition
Broken Parking Meter
Building Condition
Building Drinking Water Tank
Building Marshal's Office
Building/Use
Bus Stop Shelter Complaint
Bus Stop Shelter Placement
Calorie Labeling
Cannabis Retailer
Commercial Disposal Complaint
Construction Lead Dust
Construction Safety Enforcement
Consumer Complaint
Cooling Tower
Cranes and Derricks
Curb Condition
DEP Highway Condition
DEP Sidewalk Condition
DEP Street Condition
DOOR/WINDOW
DSNY Internal
Damaged Tree
Day Care
Dead Animal
Dead/Dying Tree
Dept of Investigations
Derelict Vehicles
Dirty Condition
Disorderly Youth
Drinking
Drinking Water
Drug Activity
Dumpster Complaint
E-Scooter
EL

In [18]:
sanitation_set = {
    # A) trash/collection/baskets/sweeping
    "Dirty Condition",
    "Missed Collection",
    "Litter Basket Complaint",
    "Litter Basket Request",
    "Recycling Basket Complaint",
    "Residential Disposal Complaint",
    "Commercial Disposal Complaint",
    "Institution Disposal Complaint",
    "Dumpster Complaint",
    "Illegal Dumping",
    "Street Sweeping Complaint",
    "Sanitation Worker or Vehicle Complaint",
    "Transfer Station Complaint",
    "DSNY Internal",

    # B) sewer/drain/wastewater
    "Sewer",
    "Root/Sewer/Sidewalk Condition",
    "Indoor Sewage",
    "Industrial Waste",
    "WATER LEAK",
    "Water System",

    # C) flooding/drainage/infra
    "Standing Water",
    "Water Conservation",
    "Water Quality",
    "Curb Condition",
    "Street Condition",
    "Highway Condition",
    "DEP Street Condition",
    "DEP Sidewalk Condition",
    "DEP Highway Condition",
    "Sidewalk Condition",

    # D) hygiene / pests
    "Rodent",
    "Mosquitoes",
    "Dead Animal",
    "UNSANITARY CONDITION",
    "Public Toilet",
    "Urinating in Public",
    "Unsanitary Pigeon Condition",
    "Unsanitary Animal Pvt Property",
    "Unsanitary Animal Facility",

    # E) blockage risk / environment
    "Overgrown Tree/Branches",
    "Wood Pile Remaining",
    "Lot Condition",
    "Obstruction",
    "Abandoned Vehicle",
    "Derelict Vehicles",
}

problem_col = "Problem (formerly Complaint Type)"

df_sanitation = df_2025[df_2025[problem_col].isin(sanitation_set)].copy()
print("df_sanitation shape:", df_sanitation.shape)

# 看看哪些类别贡献最大
print(df_sanitation[problem_col].value_counts().head(30))

# 顺便检查：sanitation_set 里有没有你数据里不存在的值（避免拼写差异）
missing = sorted([x for x in sanitation_set if x not in set(df_2025[problem_col].dropna().unique())])
print("Values in sanitation_set but NOT in data:", missing)

df_sanitation shape: (730355, 44)
Problem (formerly Complaint Type)
UNSANITARY CONDITION                      90418
Water System                              77517
Street Condition                          70330
Abandoned Vehicle                         67903
Dirty Condition                           61047
Derelict Vehicles                         43229
Missed Collection                         42977
Illegal Dumping                           39002
Rodent                                    31312
WATER LEAK                                30331
Sewer                                     26141
Sidewalk Condition                        25809
Obstruction                               19836
Overgrown Tree/Branches                   19090
Residential Disposal Complaint            16392
Dead Animal                               10711
Root/Sewer/Sidewalk Condition             10446
Curb Condition                             8070
Street Sweeping Complaint                  7257
Litter Basket Reques

In [20]:
df_sanitation.to_csv("311_2025_sanitation.csv", index=False)